# Airbnb Prices in European Cities — Data Preparation (Steps 1–3)
Load & merge → detect missing values → collect extra features (Selenium + Inside Airbnb) → handle missing values → save cleaned Excel.

Extra features source: [Inside Airbnb](https://insideairbnb.com/get-the-data/), licensed CC BY 4.0.

## Step 1 — Connect Dataset Files
### Code 1: Loading All 20 City CSV Files

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import json

folder = r"C:\Projects\airbnb-prices-europe\data"
files = sorted(glob.glob(os.path.join(folder, "*.csv")))

print(len(files))
print([os.path.basename(f) for f in files[:4]])

20
['amsterdam_weekdays.csv', 'amsterdam_weekends.csv', 'athens_weekdays.csv', 'athens_weekends.csv']


### Code 2: Adding City and Day-Type Labels from File Names

In [2]:
frames = []

for f in files:
    name = os.path.basename(f).replace(".csv", "")
    city, day_type = name.split("_")
    df = pd.read_csv(f)
    df["city"] = city.title()
    df["day_type"] = day_type.title()
    frames.append(df)

first_cols = set(frames[0].columns)
print(all(set(df.columns) == first_cols for df in frames))

True


### Code 3: Building the Master Dataset

In [3]:
master = pd.concat(frames, ignore_index=True)
master = master.drop(columns=["Unnamed: 0"])

print(master.shape)
master.head(3)

(51707, 21)


,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,Weekdays
1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,Weekdays
2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,Weekdays


### Code 4: Master Dataset Sanity Check

In [4]:
print(master["city"].value_counts())
print(master["day_type"].value_counts())
print("Total nulls:", master.isnull().sum().sum())
print("Duplicate rows:", master.duplicated().sum())

city
London       9993
Rome         9027
Paris        6688
Lisbon       5763
Athens       5280
Budapest     4022
Vienna       3537
Barcelona    2833
Berlin       2484
Amsterdam    2080
Name: count, dtype: int64
day_type
Weekends    26207
Weekdays    25500
Name: count, dtype: int64
Total nulls: 0
Duplicate rows: 0


## Step 2 — Handle Missing Values
### 2A: Null Values per Column

In [5]:
print(master.isnull().sum())

realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
city                          0
day_type                      0
dtype: int64


### 2B: Price Correlation Check (original features)

In [6]:
price_corr = master.corr(numeric_only=True)["realSum"]
price_corr = price_corr.drop("realSum").sort_values(ascending=False)
print(price_corr.round(2))

attr_index_norm               0.29
bedrooms                      0.22
lat                           0.21
person_capacity               0.20
attr_index                    0.18
rest_index_norm               0.14
rest_index                    0.13
biz                           0.04
guest_satisfaction_overall   -0.00
cleanliness_rating           -0.01
room_shared                  -0.04
dist                         -0.04
host_is_superhost            -0.05
multi                        -0.05
metro_dist                   -0.06
room_private                 -0.17
lng                          -0.17
Name: realSum, dtype: float64


All correlations are weak (|r| < 0.3), so extra features are needed (slide 7).

## 2C: Collecting Extra Features
Airbnb's terms prohibit scraping and the dataset has no listing IDs, so Selenium collects the download links from Inside Airbnb's public data page instead.
### Block 1: Collecting Download Links with Selenium

In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By

cities = master["city"].unique()

driver = webdriver.Edge()
driver.get("https://insideairbnb.com/get-the-data/")
all_links = driver.find_elements(By.TAG_NAME, "a")
all_hrefs = [link.get_attribute("href") for link in all_links]
driver.quit()

listing_urls = {}
for city in cities:
    for href in all_hrefs:
        if href and f"/{city.lower()}/" in href and href.endswith("/data/listings.csv.gz"):
            listing_urls[city] = href
            break

for city, url in listing_urls.items():
    print(city, "->", url)

Amsterdam -> https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2026-09-16/data/listings.csv.gz
Athens -> https://data.insideairbnb.com/greece/attica/athens/2026-09-22/data/listings.csv.gz
Barcelona -> https://data.insideairbnb.com/spain/catalonia/barcelona/2026-09-17/data/listings.csv.gz
Berlin -> https://data.insideairbnb.com/germany/be/berlin/2026-09-19/data/listings.csv.gz
Budapest -> https://data.insideairbnb.com/hungary/k%C3%B6z%C3%A9p-magyarorsz%C3%A1g/budapest/2026-09-21/data/listings.csv.gz
Lisbon -> https://data.insideairbnb.com/portugal/lisbon/lisbon/2026-09-19/data/listings.csv.gz
London -> https://data.insideairbnb.com/united-kingdom/england/london/2026-09-17/data/listings.csv.gz
Paris -> https://data.insideairbnb.com/france/ile-de-france/paris/2026-09-16/data/listings.csv.gz
Rome -> https://data.insideairbnb.com/italy/lazio/rome/2026-09-17/data/listings.csv.gz
Vienna -> https://data.insideairbnb.com/austria/vienna/vienna/2026-09-17/data/listings.csv.gz


### Block 2: Downloading the Inside Airbnb Listing Files

In [8]:
import requests

save_folder = os.path.join(folder, "insideairbnb")
os.makedirs(save_folder, exist_ok=True)
headers = {"User-Agent": "Mozilla/5.0"}

for city, url in listing_urls.items():
    path = os.path.join(save_folder, f"{city.lower()}_listings.csv.gz")
    if not os.path.exists(path):
        response = requests.get(url, headers=headers, timeout=600)
        response.raise_for_status()
        with open(path, "wb") as file:
            file.write(response.content)
    print(city, "saved")

Amsterdam saved


Athens saved


Barcelona saved


Berlin saved


Budapest saved


Lisbon saved


London saved


Paris saved


Rome saved


Vienna saved


### Block 3: Loading the Extra Features

In [9]:
keep_cols = ["latitude", "longitude", "room_type", "property_type", "neighbourhood_cleansed",
             "amenities", "number_of_reviews", "review_scores_rating", "availability_365"]

extra_frames = []
for city in listing_urls:
    path = os.path.join(save_folder, f"{city.lower()}_listings.csv.gz")
    df = pd.read_csv(path, usecols=keep_cols)
    df["city"] = city
    extra_frames.append(df)

extra = pd.concat(extra_frames, ignore_index=True)
extra["amenities_count"] = extra["amenities"].apply(lambda x: len(json.loads(x)) if isinstance(x, str) else np.nan)
extra = extra.drop(columns=["amenities"])
extra = extra.rename(columns={"neighbourhood_cleansed": "neighbourhood"})

print(extra.shape)
print(extra["city"].value_counts())
extra.head(3)

(309154, 10)
city
London       91954
Paris        77149
Rome         37451
Lisbon       25020
Barcelona    15236
Athens       15134
Vienna       12982
Berlin       12376
Budapest     11240
Amsterdam    10612
Name: count, dtype: int64


,neighbourhood,latitude,longitude,property_type,room_type,availability_365,number_of_reviews,review_scores_rating,city,amenities_count
0,Centrum-West,52.36775,4.89092,Private room in rental unit,Private room,49,815,4.86,Amsterdam,18
1,Centrum-Oost,52.36584,4.89111,Private room in condo,Private room,0,929,4.82,Amsterdam,18
2,Centrum-Oost,52.37168,4.91471,Entire rental unit,Entire home/apt,0,42,4.71,Amsterdam,31


### Block 4: Matching Each Listing to Its Nearest Inside Airbnb Listing

In [10]:
from sklearn.neighbors import BallTree

feature_cols = ["property_type", "neighbourhood", "amenities_count",
                "number_of_reviews", "review_scores_rating", "availability_365"]

matched_parts = []
for (city, room), group in master.groupby(["city", "room_type"]):
    candidates = extra[(extra["city"] == city) & (extra["room_type"] == room)]
    if len(candidates) == 0:
        continue
    tree = BallTree(np.radians(candidates[["latitude", "longitude"]]), metric="haversine")
    distance, position = tree.query(np.radians(group[["lat", "lng"]]), k=1)
    nearest = candidates.iloc[position[:, 0]].copy()
    nearest["match_distance_m"] = distance[:, 0] * 6371000
    nearest.index = group.index
    matched_parts.append(nearest)

matched = pd.concat(matched_parts)
too_far = matched["match_distance_m"] > 200
matched[feature_cols] = matched[feature_cols].where(~too_far, axis=0)

master_enriched = master.join(matched[feature_cols + ["match_distance_m"]])
master_enriched["is_matched"] = master_enriched["match_distance_m"] <= 200

print(master_enriched.shape)
print("Match rate:", master_enriched["is_matched"].mean().round(3))
print(master_enriched.groupby("city")["is_matched"].mean().round(3))

(51707, 29)
Match rate: 0.969
city
Amsterdam    0.922
Athens       0.984
Barcelona    0.961
Berlin       0.862
Budapest     0.944
Lisbon       0.980
London       0.987
Paris        0.990
Rome         0.989
Vienna       0.925
Name: is_matched, dtype: float64


### Block 5: Detecting and Handling the New Missing Values

In [11]:
print(master_enriched[feature_cols].isnull().sum())

numeric_cols = ["amenities_count", "number_of_reviews", "review_scores_rating", "availability_365"]
for col in numeric_cols:
    city_median = master_enriched.groupby("city")[col].transform("median")
    master_enriched[col] = master_enriched[col].fillna(city_median)

for col in ["property_type", "neighbourhood"]:
    master_enriched[col] = master_enriched[col].fillna("Unknown")

print("Nulls left:", master_enriched[feature_cols].isnull().sum().sum())

property_type           1599
neighbourhood           1599
amenities_count         1599
number_of_reviews       1599
review_scores_rating    9151
availability_365        1599
dtype: int64
Nulls left: 0


### Block 6: Re-checking Price Correlation with the New Features

In [12]:
new_corr = master_enriched.corr(numeric_only=True)["realSum"]
new_corr = new_corr.drop("realSum").sort_values(ascending=False)
print(new_corr.round(2))

attr_index_norm               0.29
bedrooms                      0.22
lat                           0.21
person_capacity               0.20
attr_index                    0.18
rest_index_norm               0.14
rest_index                    0.13
is_matched                    0.05
biz                           0.04
amenities_count               0.03
review_scores_rating          0.01
guest_satisfaction_overall   -0.00
cleanliness_rating           -0.01
number_of_reviews            -0.02
room_shared                  -0.04
availability_365             -0.04
dist                         -0.04
host_is_superhost            -0.05
match_distance_m             -0.05
multi                        -0.05
metro_dist                   -0.06
room_private                 -0.17
lng                          -0.17
Name: realSum, dtype: float64


## Step 3 — Save Cleaned Data

In [13]:
output_path = r"C:\Projects\airbnb-prices-europe\airbnb_cleaned.xlsx"
master_enriched.to_excel(output_path, index=False)
print("Saved:", output_path, master_enriched.shape)

Saved: C:\Projects\airbnb-prices-europe\airbnb_cleaned.xlsx (51707, 29)
